In [1]:
import os
import pandas as pd
from PIL import Image
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
from reportlab.lib.utils import ImageReader
import io

**The following code segment is for all the data**

In [2]:
methods = ['v1', 'robust', 'synthstrip', 'hdctbet', 'ctbet', 'brainchop']
all_results = []

brainchop_exclude = [
    '6109-317_20150302_0647_ct.png',
    '6142-308_20150610_0707_ct.png', 
    '6193-324_20150924_1431_ct.png',
    '6257-335_20160118_1150_ct.png',
    '6418-193_20161228_1248_ct.png',
    '6470-296_20170602_0607_ct.png',
    '6480-154_20170622_0937_ct.png'
]

for method in methods:
    print(f"Processing method: {method}")
    original_csv = pd.read_csv(f"/Users/rushil/ichseg/{method}/annotations.csv")
    failures_csv = pd.read_csv(f"/Users/rushil/ichseg/{method}/annotations_failures.csv")
    
    if method == 'brainchop':
        original_csv = original_csv[~original_csv['Filename'].isin(brainchop_exclude)]
        failures_csv = failures_csv[~failures_csv['Filename'].isin(brainchop_exclude)]
        print(f"Filtered out {len(brainchop_exclude)} files from brainchop")

    all_columns_to_check = ['1 - Neck', '3 - Holes', '4 - Nonbrain']
    columns_to_check = [col for col in all_columns_to_check if col in original_csv.columns and col in failures_csv.columns]
    print(f"Available columns for {method}: {columns_to_check}")
    
    original_count = len(original_csv)
    if method == 'robust':
        p1_failures_count = len(original_csv[original_csv[columns_to_check].eq('yes').any(axis=1)]) if columns_to_check else 0
    else:
        p1_failures_count = len(failures_csv)

    p2_failures_count = len(failures_csv[failures_csv[columns_to_check].eq('yes').any(axis=1)]) if columns_to_check else 0
    
    # Individual column counts from original CSV (Pass 1)
    original_neck_count = len(original_csv[original_csv['1 - Neck'] == 'yes']) if '1 - Neck' in columns_to_check else 0
    original_holes_count = len(original_csv[original_csv['3 - Holes'] == 'yes']) if '3 - Holes' in columns_to_check else 0
    original_nonbrain_count = len(original_csv[original_csv['4 - Nonbrain'] == 'yes']) if '4 - Nonbrain' in columns_to_check else 0
    
    # Multiple failures count from original CSV (Pass 1)
    yes_counts_per_row_original = original_csv[columns_to_check].eq('yes').sum(axis=1) if columns_to_check else pd.Series([0] * len(original_csv))
    original_multiple_failures_count = len(original_csv[yes_counts_per_row_original >= 2])
    
    # Individual column counts from failures CSV (Pass 2)
    neck_count = len(failures_csv[failures_csv['1 - Neck'] == 'yes']) if '1 - Neck' in columns_to_check else 0
    holes_count = len(failures_csv[failures_csv['3 - Holes'] == 'yes']) if '3 - Holes' in columns_to_check else 0
    nonbrain_count = len(failures_csv[failures_csv['4 - Nonbrain'] == 'yes']) if '4 - Nonbrain' in columns_to_check else 0
    
    # Multiple failures count from failures CSV (Pass 2)
    yes_counts_per_row_failures = failures_csv[columns_to_check].eq('yes').sum(axis=1) if columns_to_check else pd.Series([0] * len(failures_csv))
    failures_multiple_failures_count = len(failures_csv[yes_counts_per_row_failures >= 2])
    
    df = pd.DataFrame({
        'Method': [method],
        'Total_Count': [original_count],
        'Pass_1_Failure_Count': [p1_failures_count],
        'Pass_1_Failure_Rate': [p1_failures_count / original_count * 100 if original_count > 0 else 0],
        'Pass_2_Failure_Count': [p2_failures_count],
        'Pass_2_Failure_Rate': [p2_failures_count / original_count * 100 if original_count > 0 else 0],
        'Pass_1_Neck_Count': [original_neck_count],
        'Pass_1_Neck_Rate': [original_neck_count / original_count * 100 if original_count > 0 else 0],
        'Pass_1_Holes_Count': [original_holes_count],
        'Pass_1_Holes_Rate': [original_holes_count / original_count * 100 if original_count > 0 else 0],
        'Pass_1_Nonbrain_Count': [original_nonbrain_count],
        'Pass_1_Nonbrain_Rate': [original_nonbrain_count / original_count * 100 if original_count > 0 else 0],
        'Pass_1_Multiple_Failures_Count': [original_multiple_failures_count],
        'Pass_1_Multiple_Failures_Rate': [original_multiple_failures_count / original_count * 100 if original_count > 0 else 0],
        'Pass_2_Neck_Count': [neck_count],
        'Pass_2_Neck_Rate': [neck_count / original_count * 100 if original_count > 0 else 0],
        'Pass_2_Holes_Count': [holes_count],
        'Pass_2_Holes_Rate': [holes_count / original_count * 100 if original_count > 0 else 0],
        'Pass_2_Nonbrain_Count': [nonbrain_count],
        'Pass_2_Nonbrain_Rate': [nonbrain_count / original_count * 100 if original_count > 0 else 0],
        'Pass_2_Multiple_Failures_Count': [failures_multiple_failures_count],
        'Pass_2_Multiple_Failures_Rate': [failures_multiple_failures_count / original_count * 100 if original_count > 0 else 0]
    })
    
    all_results.append(df)
    
final_results = pd.concat(all_results, ignore_index=True)
column_order = [
    "Method",
    "Total_Count",
    "Pass_1_Failure_Count", "Pass_1_Failure_Rate",
    "Pass_1_Neck_Count",    "Pass_1_Neck_Rate",
    "Pass_1_Holes_Count",   "Pass_1_Holes_Rate",
    "Pass_1_Nonbrain_Count","Pass_1_Nonbrain_Rate",
    "Pass_1_Multiple_Failures_Count","Pass_1_Multiple_Failures_Rate",
    "Pass_2_Failure_Count", "Pass_2_Failure_Rate",
    "Pass_2_Neck_Count",           "Pass_2_Neck_Rate",
    "Pass_2_Holes_Count",          "Pass_2_Holes_Rate",
    "Pass_2_Nonbrain_Count",       "Pass_2_Nonbrain_Rate",
    "Pass_2_Multiple_Failures_Count","Pass_2_Multiple_Failures_Rate"
]

final_results = final_results[column_order]
#final_results
final_results.to_csv('/Users/rushil/ichseg/local_results/Rushil_QC_results.csv', index=False)

Processing method: v1
Available columns for v1: ['1 - Neck', '3 - Holes', '4 - Nonbrain']
Processing method: robust
Available columns for robust: ['1 - Neck', '3 - Holes', '4 - Nonbrain']
Processing method: synthstrip
Available columns for synthstrip: ['1 - Neck', '3 - Holes', '4 - Nonbrain']
Processing method: hdctbet
Available columns for hdctbet: ['1 - Neck', '3 - Holes', '4 - Nonbrain']
Processing method: ctbet
Available columns for ctbet: ['1 - Neck', '3 - Holes', '4 - Nonbrain']
Processing method: brainchop
Filtered out 7 files from brainchop
Available columns for brainchop: ['1 - Neck', '3 - Holes', '4 - Nonbrain']


**The following code segment is for Crainotomy data**

In [8]:
methods = ['v1', 'robust', 'synthstrip', 'hdctbet', 'ctbet', 'brainchop']
all_results = []

brainchop_exclude = [
    '6109-317_20150302_0647_ct.png',
    '6142-308_20150610_0707_ct.png', 
    '6193-324_20150924_1431_ct.png',
    '6257-335_20160118_1150_ct.png',
    '6418-193_20161228_1248_ct.png',
    '6470-296_20170602_0607_ct.png',
    '6480-154_20170622_0937_ct.png'
]

for method in methods:
    print(f"Processing method: {method}")
    original_csv = pd.read_csv(f"/Users/rushil/ichseg/{method}/annotations.csv")
    failures_csv = pd.read_csv(f"/Users/rushil/ichseg/{method}/annotations_failures.csv")
    
    if method == 'brainchop':
        original_csv = original_csv[~original_csv['Filename'].isin(brainchop_exclude)]
        failures_csv = failures_csv[~failures_csv['Filename'].isin(brainchop_exclude)]
        print(f"Filtered out {len(brainchop_exclude)} files from brainchop")

    # Filter for craniotomy cases only
    if '7 - Craniotomy' in original_csv.columns:
        original_csv = original_csv[original_csv['7 - Craniotomy'] == 'yes']
        print(f"Filtered to craniotomy cases: {len(original_csv)} remaining")
    else:
        print(f"No '7 - Craniotomy' column found, using all {len(original_csv)} cases")
    
    if '7 - Craniotomy' in failures_csv.columns:
        failures_csv = failures_csv[failures_csv['7 - Craniotomy'] == 'yes']
        print(f"Filtered failures to craniotomy cases: {len(failures_csv)} remaining")

    all_columns_to_check = ['1 - Neck', '3 - Holes', '4 - Nonbrain']
    columns_to_check = [col for col in all_columns_to_check if col in original_csv.columns and col in failures_csv.columns]
    print(f"Available columns for {method}: {columns_to_check}")
    
    original_count = len(original_csv)
    
    # Calculate p1_failures_count AFTER filtering
    if method == 'robust':
        p1_failures_count = len(original_csv[original_csv[columns_to_check].eq('yes').any(axis=1)]) if columns_to_check else 0
    else:
        p1_failures_count = len(failures_csv)  # Now using filtered failures_csv

    p2_failures_count = len(failures_csv[failures_csv[columns_to_check].eq('yes').any(axis=1)]) if columns_to_check else 0
    
    # Individual column counts from original CSV (Pass 1)
    original_neck_count = len(original_csv[original_csv['1 - Neck'] == 'yes']) if '1 - Neck' in columns_to_check else 0
    original_holes_count = len(original_csv[original_csv['3 - Holes'] == 'yes']) if '3 - Holes' in columns_to_check else 0
    original_nonbrain_count = len(original_csv[original_csv['4 - Nonbrain'] == 'yes']) if '4 - Nonbrain' in columns_to_check else 0
    
    # Multiple failures count from original CSV (Pass 1)
    yes_counts_per_row_original = original_csv[columns_to_check].eq('yes').sum(axis=1) if columns_to_check else pd.Series([0] * len(original_csv))
    original_multiple_failures_count = len(original_csv[yes_counts_per_row_original >= 2])
    
    # Individual column counts from failures CSV (Pass 2)
    neck_count = len(failures_csv[failures_csv['1 - Neck'] == 'yes']) if '1 - Neck' in columns_to_check else 0
    holes_count = len(failures_csv[failures_csv['3 - Holes'] == 'yes']) if '3 - Holes' in columns_to_check else 0
    nonbrain_count = len(failures_csv[failures_csv['4 - Nonbrain'] == 'yes']) if '4 - Nonbrain' in columns_to_check else 0
    
    # Multiple failures count from failures CSV (Pass 2)
    yes_counts_per_row_failures = failures_csv[columns_to_check].eq('yes').sum(axis=1) if columns_to_check else pd.Series([0] * len(failures_csv))
    failures_multiple_failures_count = len(failures_csv[yes_counts_per_row_failures >= 2])
    
    df = pd.DataFrame({
        'Method': [method],
        'Total_Count': [original_count],
        'Pass_1_Failure_Count': [p1_failures_count],
        'Pass_1_Failure_Rate': [p1_failures_count / original_count * 100 if original_count > 0 else 0],
        'Pass_2_Failure_Count': [p2_failures_count],
        'Pass_2_Failure_Rate': [p2_failures_count / original_count * 100 if original_count > 0 else 0],
        'Pass_1_Neck_Count': [original_neck_count],
        'Pass_1_Neck_Rate': [original_neck_count / original_count * 100 if original_count > 0 else 0],
        'Pass_1_Holes_Count': [original_holes_count],
        'Pass_1_Holes_Rate': [original_holes_count / original_count * 100 if original_count > 0 else 0],
        'Pass_1_Nonbrain_Count': [original_nonbrain_count],
        'Pass_1_Nonbrain_Rate': [original_nonbrain_count / original_count * 100 if original_count > 0 else 0],
        'Pass_1_Multiple_Failures_Count': [original_multiple_failures_count],
        'Pass_1_Multiple_Failures_Rate': [original_multiple_failures_count / original_count * 100 if original_count > 0 else 0],
        'Pass_2_Neck_Count': [neck_count],
        'Pass_2_Neck_Rate': [neck_count / original_count * 100 if original_count > 0 else 0],
        'Pass_2_Holes_Count': [holes_count],
        'Pass_2_Holes_Rate': [holes_count / original_count * 100 if original_count > 0 else 0],
        'Pass_2_Nonbrain_Count': [nonbrain_count],
        'Pass_2_Nonbrain_Rate': [nonbrain_count / original_count * 100 if original_count > 0 else 0],
        'Pass_2_Multiple_Failures_Count': [failures_multiple_failures_count],
        'Pass_2_Multiple_Failures_Rate': [failures_multiple_failures_count / original_count * 100 if original_count > 0 else 0]
    })
    
    all_results.append(df)
    
final_results = pd.concat(all_results, ignore_index=True)
column_order = [
    "Method",
    "Total_Count",
    "Pass_1_Failure_Count", "Pass_1_Failure_Rate",
    "Pass_1_Neck_Count",    "Pass_1_Neck_Rate",
    "Pass_1_Holes_Count",   "Pass_1_Holes_Rate",
    "Pass_1_Nonbrain_Count","Pass_1_Nonbrain_Rate",
    "Pass_1_Multiple_Failures_Count","Pass_1_Multiple_Failures_Rate",
    "Pass_2_Failure_Count", "Pass_2_Failure_Rate",
    "Pass_2_Neck_Count",           "Pass_2_Neck_Rate",
    "Pass_2_Holes_Count",          "Pass_2_Holes_Rate",
    "Pass_2_Nonbrain_Count",       "Pass_2_Nonbrain_Rate",
    "Pass_2_Multiple_Failures_Count","Pass_2_Multiple_Failures_Rate"
]

final_results = final_results[column_order]
#final_results
final_results.to_csv('/Users/rushil/ichseg/local_results/Rushil_QC_crainotomy_results.csv', index=False)

Processing method: v1
Filtered to craniotomy cases: 40 remaining
Filtered failures to craniotomy cases: 35 remaining
Available columns for v1: ['1 - Neck', '3 - Holes', '4 - Nonbrain']
Processing method: robust
Filtered to craniotomy cases: 40 remaining
Filtered failures to craniotomy cases: 0 remaining
Available columns for robust: ['1 - Neck', '3 - Holes', '4 - Nonbrain']
Processing method: synthstrip
Filtered to craniotomy cases: 40 remaining
Filtered failures to craniotomy cases: 38 remaining
Available columns for synthstrip: ['1 - Neck', '3 - Holes', '4 - Nonbrain']
Processing method: hdctbet
Filtered to craniotomy cases: 40 remaining
Filtered failures to craniotomy cases: 40 remaining
Available columns for hdctbet: ['1 - Neck', '3 - Holes', '4 - Nonbrain']
Processing method: ctbet
Filtered to craniotomy cases: 22 remaining
Filtered failures to craniotomy cases: 21 remaining
Available columns for ctbet: ['1 - Neck', '3 - Holes', '4 - Nonbrain']
Processing method: brainchop
Filtere

**The following code segment is for CTA data**

In [9]:
methods = ['v1', 'robust', 'synthstrip', 'hdctbet', 'ctbet', 'brainchop']
all_results = []

brainchop_exclude = [
    '6109-317_20150302_0647_ct.png',
    '6142-308_20150610_0707_ct.png', 
    '6193-324_20150924_1431_ct.png',
    '6257-335_20160118_1150_ct.png',
    '6418-193_20161228_1248_ct.png',
    '6470-296_20170602_0607_ct.png',
    '6480-154_20170622_0937_ct.png'
]

for method in methods:
    print(f"Processing method: {method}")
    original_csv = pd.read_csv(f"/Users/rushil/ichseg/{method}/annotations.csv")
    failures_csv = pd.read_csv(f"/Users/rushil/ichseg/{method}/annotations_failures.csv")
    
    if method == 'brainchop':
        original_csv = original_csv[~original_csv['Filename'].isin(brainchop_exclude)]
        failures_csv = failures_csv[~failures_csv['Filename'].isin(brainchop_exclude)]
        print(f"Filtered out {len(brainchop_exclude)} files from brainchop")

    # Filter for cta cases only
    if '5 - CTA' in original_csv.columns:
        original_csv = original_csv[original_csv['5 - CTA'] == 'yes']
        print(f"Filtered to cta cases: {len(original_csv)} remaining")
    else:
        print(f"No '5 - CTA' column found, using all {len(original_csv)} cases")
    
    if '5 - CTA' in failures_csv.columns:
        failures_csv = failures_csv[failures_csv['5 - CTA'] == 'yes']
        print(f"Filtered failures to cta cases: {len(failures_csv)} remaining")

    all_columns_to_check = ['1 - Neck', '3 - Holes', '4 - Nonbrain']
    columns_to_check = [col for col in all_columns_to_check if col in original_csv.columns and col in failures_csv.columns]
    print(f"Available columns for {method}: {columns_to_check}")
    
    original_count = len(original_csv)
    
    # Calculate p1_failures_count AFTER filtering
    if method == 'robust':
        p1_failures_count = len(original_csv[original_csv[columns_to_check].eq('yes').any(axis=1)]) if columns_to_check else 0
    else:
        p1_failures_count = len(failures_csv)  # Now using filtered failures_csv

    p2_failures_count = len(failures_csv[failures_csv[columns_to_check].eq('yes').any(axis=1)]) if columns_to_check else 0
    
    # Individual column counts from original CSV (Pass 1)
    original_neck_count = len(original_csv[original_csv['1 - Neck'] == 'yes']) if '1 - Neck' in columns_to_check else 0
    original_holes_count = len(original_csv[original_csv['3 - Holes'] == 'yes']) if '3 - Holes' in columns_to_check else 0
    original_nonbrain_count = len(original_csv[original_csv['4 - Nonbrain'] == 'yes']) if '4 - Nonbrain' in columns_to_check else 0
    
    # Multiple failures count from original CSV (Pass 1)
    yes_counts_per_row_original = original_csv[columns_to_check].eq('yes').sum(axis=1) if columns_to_check else pd.Series([0] * len(original_csv))
    original_multiple_failures_count = len(original_csv[yes_counts_per_row_original >= 2])
    
    # Individual column counts from failures CSV (Pass 2)
    neck_count = len(failures_csv[failures_csv['1 - Neck'] == 'yes']) if '1 - Neck' in columns_to_check else 0
    holes_count = len(failures_csv[failures_csv['3 - Holes'] == 'yes']) if '3 - Holes' in columns_to_check else 0
    nonbrain_count = len(failures_csv[failures_csv['4 - Nonbrain'] == 'yes']) if '4 - Nonbrain' in columns_to_check else 0
    
    # Multiple failures count from failures CSV (Pass 2)
    yes_counts_per_row_failures = failures_csv[columns_to_check].eq('yes').sum(axis=1) if columns_to_check else pd.Series([0] * len(failures_csv))
    failures_multiple_failures_count = len(failures_csv[yes_counts_per_row_failures >= 2])
    
    df = pd.DataFrame({
        'Method': [method],
        'Total_Count': [original_count],
        'Pass_1_Failure_Count': [p1_failures_count],
        'Pass_1_Failure_Rate': [p1_failures_count / original_count * 100 if original_count > 0 else 0],
        'Pass_2_Failure_Count': [p2_failures_count],
        'Pass_2_Failure_Rate': [p2_failures_count / original_count * 100 if original_count > 0 else 0],
        'Pass_1_Neck_Count': [original_neck_count],
        'Pass_1_Neck_Rate': [original_neck_count / original_count * 100 if original_count > 0 else 0],
        'Pass_1_Holes_Count': [original_holes_count],
        'Pass_1_Holes_Rate': [original_holes_count / original_count * 100 if original_count > 0 else 0],
        'Pass_1_Nonbrain_Count': [original_nonbrain_count],
        'Pass_1_Nonbrain_Rate': [original_nonbrain_count / original_count * 100 if original_count > 0 else 0],
        'Pass_1_Multiple_Failures_Count': [original_multiple_failures_count],
        'Pass_1_Multiple_Failures_Rate': [original_multiple_failures_count / original_count * 100 if original_count > 0 else 0],
        'Pass_2_Neck_Count': [neck_count],
        'Pass_2_Neck_Rate': [neck_count / original_count * 100 if original_count > 0 else 0],
        'Pass_2_Holes_Count': [holes_count],
        'Pass_2_Holes_Rate': [holes_count / original_count * 100 if original_count > 0 else 0],
        'Pass_2_Nonbrain_Count': [nonbrain_count],
        'Pass_2_Nonbrain_Rate': [nonbrain_count / original_count * 100 if original_count > 0 else 0],
        'Pass_2_Multiple_Failures_Count': [failures_multiple_failures_count],
        'Pass_2_Multiple_Failures_Rate': [failures_multiple_failures_count / original_count * 100 if original_count > 0 else 0]
    })
    
    all_results.append(df)
    
final_results = pd.concat(all_results, ignore_index=True)
column_order = [
    "Method",
    "Total_Count",
    "Pass_1_Failure_Count", "Pass_1_Failure_Rate",
    "Pass_1_Neck_Count",    "Pass_1_Neck_Rate",
    "Pass_1_Holes_Count",   "Pass_1_Holes_Rate",
    "Pass_1_Nonbrain_Count","Pass_1_Nonbrain_Rate",
    "Pass_1_Multiple_Failures_Count","Pass_1_Multiple_Failures_Rate",
    "Pass_2_Failure_Count", "Pass_2_Failure_Rate",
    "Pass_2_Neck_Count",           "Pass_2_Neck_Rate",
    "Pass_2_Holes_Count",          "Pass_2_Holes_Rate",
    "Pass_2_Nonbrain_Count",       "Pass_2_Nonbrain_Rate",
    "Pass_2_Multiple_Failures_Count","Pass_2_Multiple_Failures_Rate"
]

final_results = final_results[column_order]
#final_results
final_results.to_csv('/Users/rushil/ichseg/local_results/Rushil_QC_CTA_results.csv', index=False)

Processing method: v1
Filtered to cta cases: 10 remaining
Filtered failures to cta cases: 6 remaining
Available columns for v1: ['1 - Neck', '3 - Holes', '4 - Nonbrain']
Processing method: robust
Filtered to cta cases: 10 remaining
Filtered failures to cta cases: 0 remaining
Available columns for robust: ['1 - Neck', '3 - Holes', '4 - Nonbrain']
Processing method: synthstrip
Filtered to cta cases: 10 remaining
Filtered failures to cta cases: 9 remaining
Available columns for synthstrip: ['1 - Neck', '3 - Holes', '4 - Nonbrain']
Processing method: hdctbet
Filtered to cta cases: 10 remaining
Filtered failures to cta cases: 10 remaining
Available columns for hdctbet: ['1 - Neck', '3 - Holes', '4 - Nonbrain']
Processing method: ctbet
Filtered to cta cases: 7 remaining
Filtered failures to cta cases: 7 remaining
Available columns for ctbet: ['1 - Neck', '3 - Holes', '4 - Nonbrain']
Processing method: brainchop
Filtered out 7 files from brainchop
Filtered to cta cases: 10 remaining
Filtered 

**The following code segment is for Artifact data**

In [7]:
import pandas as pd
methods = ['v1', 'robust', 'synthstrip', 'hdctbet', 'ctbet', 'brainchop']
all_results = []

brainchop_exclude = [
    '6109-317_20150302_0647_ct.png',
    '6142-308_20150610_0707_ct.png', 
    '6193-324_20150924_1431_ct.png',
    '6257-335_20160118_1150_ct.png',
    '6418-193_20161228_1248_ct.png',
    '6470-296_20170602_0607_ct.png',
    '6480-154_20170622_0937_ct.png'
]

for method in methods:
    print(f"Processing method: {method}")
    original_csv = pd.read_csv(f"/Users/rushil/ichseg/{method}/annotations.csv")
    failures_csv = pd.read_csv(f"/Users/rushil/ichseg/{method}/annotations_failures.csv")
    
    if method == 'brainchop':
        original_csv = original_csv[~original_csv['Filename'].isin(brainchop_exclude)]
        failures_csv = failures_csv[~failures_csv['Filename'].isin(brainchop_exclude)]
        print(f"Filtered out {len(brainchop_exclude)} files from brainchop")

    # Filter for artifact cases only
    if '6 - Noisy Artifacts' in original_csv.columns:
        original_csv = original_csv[original_csv['6 - Noisy Artifacts'] == 'yes']
        print(f"Filtered to artifact cases: {len(original_csv)} remaining")
    else:
        print(f"No '6 - Noisy Artifacts' column found, using all {len(original_csv)} cases")
    
    if '6 - Noisy Artifacts' in failures_csv.columns:
        failures_csv = failures_csv[failures_csv['6 - Noisy Artifacts'] == 'yes']
        print(f"Filtered failures to artifact cases: {len(failures_csv)} remaining")

    all_columns_to_check = ['1 - Neck', '3 - Holes', '4 - Nonbrain']
    columns_to_check = [col for col in all_columns_to_check if col in original_csv.columns and col in failures_csv.columns]
    print(f"Available columns for {method}: {columns_to_check}")
    
    original_count = len(original_csv)
    
    # Calculate p1_failures_count AFTER filtering
    if method == 'robust':
        p1_failures_count = len(original_csv[original_csv[columns_to_check].eq('yes').any(axis=1)]) if columns_to_check else 0
    else:
        p1_failures_count = len(failures_csv)  # Now using filtered failures_csv

    p2_failures_count = len(failures_csv[failures_csv[columns_to_check].eq('yes').any(axis=1)]) if columns_to_check else 0
    
    # Individual column counts from original CSV (Pass 1)
    original_neck_count = len(original_csv[original_csv['1 - Neck'] == 'yes']) if '1 - Neck' in columns_to_check else 0
    original_holes_count = len(original_csv[original_csv['3 - Holes'] == 'yes']) if '3 - Holes' in columns_to_check else 0
    original_nonbrain_count = len(original_csv[original_csv['4 - Nonbrain'] == 'yes']) if '4 - Nonbrain' in columns_to_check else 0
    
    # Multiple failures count from original CSV (Pass 1)
    yes_counts_per_row_original = original_csv[columns_to_check].eq('yes').sum(axis=1) if columns_to_check else pd.Series([0] * len(original_csv))
    original_multiple_failures_count = len(original_csv[yes_counts_per_row_original >= 2])
    
    # Individual column counts from failures CSV (Pass 2)
    neck_count = len(failures_csv[failures_csv['1 - Neck'] == 'yes']) if '1 - Neck' in columns_to_check else 0
    holes_count = len(failures_csv[failures_csv['3 - Holes'] == 'yes']) if '3 - Holes' in columns_to_check else 0
    nonbrain_count = len(failures_csv[failures_csv['4 - Nonbrain'] == 'yes']) if '4 - Nonbrain' in columns_to_check else 0
    
    # Multiple failures count from failures CSV (Pass 2)
    yes_counts_per_row_failures = failures_csv[columns_to_check].eq('yes').sum(axis=1) if columns_to_check else pd.Series([0] * len(failures_csv))
    failures_multiple_failures_count = len(failures_csv[yes_counts_per_row_failures >= 2])
    
    df = pd.DataFrame({
        'Method': [method],
        'Total_Count': [original_count],
        'Pass_1_Failure_Count': [p1_failures_count],
        'Pass_1_Failure_Rate': [p1_failures_count / original_count * 100 if original_count > 0 else 0],
        'Pass_2_Failure_Count': [p2_failures_count],
        'Pass_2_Failure_Rate': [p2_failures_count / original_count * 100 if original_count > 0 else 0],
        'Pass_1_Neck_Count': [original_neck_count],
        'Pass_1_Neck_Rate': [original_neck_count / original_count * 100 if original_count > 0 else 0],
        'Pass_1_Holes_Count': [original_holes_count],
        'Pass_1_Holes_Rate': [original_holes_count / original_count * 100 if original_count > 0 else 0],
        'Pass_1_Nonbrain_Count': [original_nonbrain_count],
        'Pass_1_Nonbrain_Rate': [original_nonbrain_count / original_count * 100 if original_count > 0 else 0],
        'Pass_1_Multiple_Failures_Count': [original_multiple_failures_count],
        'Pass_1_Multiple_Failures_Rate': [original_multiple_failures_count / original_count * 100 if original_count > 0 else 0],
        'Pass_2_Neck_Count': [neck_count],
        'Pass_2_Neck_Rate': [neck_count / original_count * 100 if original_count > 0 else 0],
        'Pass_2_Holes_Count': [holes_count],
        'Pass_2_Holes_Rate': [holes_count / original_count * 100 if original_count > 0 else 0],
        'Pass_2_Nonbrain_Count': [nonbrain_count],
        'Pass_2_Nonbrain_Rate': [nonbrain_count / original_count * 100 if original_count > 0 else 0],
        'Pass_2_Multiple_Failures_Count': [failures_multiple_failures_count],
        'Pass_2_Multiple_Failures_Rate': [failures_multiple_failures_count / original_count * 100 if original_count > 0 else 0]
    })
    
    all_results.append(df)
    
final_results = pd.concat(all_results, ignore_index=True)
column_order = [
    "Method",
    "Total_Count",
    "Pass_1_Failure_Count", "Pass_1_Failure_Rate",
    "Pass_1_Neck_Count",    "Pass_1_Neck_Rate",
    "Pass_1_Holes_Count",   "Pass_1_Holes_Rate",
    "Pass_1_Nonbrain_Count","Pass_1_Nonbrain_Rate",
    "Pass_1_Multiple_Failures_Count","Pass_1_Multiple_Failures_Rate",
    "Pass_2_Failure_Count", "Pass_2_Failure_Rate",
    "Pass_2_Neck_Count",           "Pass_2_Neck_Rate",
    "Pass_2_Holes_Count",          "Pass_2_Holes_Rate",
    "Pass_2_Nonbrain_Count",       "Pass_2_Nonbrain_Rate",
    "Pass_2_Multiple_Failures_Count","Pass_2_Multiple_Failures_Rate"
]

final_results = final_results[column_order]
#final_results
final_results.to_csv('/Users/rushil/ichseg/local_results/Rushil_QC_Artifact_results.csv', index=False)

Processing method: v1
Filtered to artifact cases: 54 remaining
Filtered failures to artifact cases: 43 remaining
Available columns for v1: ['1 - Neck', '3 - Holes', '4 - Nonbrain']
Processing method: robust
Filtered to artifact cases: 54 remaining
Filtered failures to artifact cases: 2 remaining
Available columns for robust: ['1 - Neck', '3 - Holes', '4 - Nonbrain']
Processing method: synthstrip
Filtered to artifact cases: 54 remaining
Filtered failures to artifact cases: 54 remaining
Available columns for synthstrip: ['1 - Neck', '3 - Holes', '4 - Nonbrain']
Processing method: hdctbet
Filtered to artifact cases: 54 remaining
Filtered failures to artifact cases: 54 remaining
Available columns for hdctbet: ['1 - Neck', '3 - Holes', '4 - Nonbrain']
Processing method: ctbet
Filtered to artifact cases: 29 remaining
Filtered failures to artifact cases: 23 remaining
Available columns for ctbet: ['1 - Neck', '3 - Holes', '4 - Nonbrain']
Processing method: brainchop
Filtered out 7 files from b